## Problem 1

We start by running the seq-to-seq example. 

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 72
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

HIDDEN_N is already 64. We change BATCH_SIZE to 64 below.

In [2]:
from ep_seq import NMTDataset, GO_token, EOS_token, PAD_token

SEQ_MXLEN= 16  # max sequence length

HIDDEN_N= 64
BATCH_SIZE= 64  # the implementation processes the sequence element by element

DATASET_SIZE = BATCH_SIZE*(131072//BATCH_SIZE)

Dataset_nmt = NMTDataset('EP_datasets/eng-fra.txt', DATASET_SIZE, SEQ_MXLEN, vectorized=True)

# sanity
print(f"#seqs={len(Dataset_nmt)} {Dataset_nmt.input_lang.name} to {Dataset_nmt.output_lang.name}")
print(f"#voc= {Dataset_nmt.input_lang.n_words} to {Dataset_nmt.output_lang.n_words}")

seq1d, seq2d = Dataset_nmt.dataset[0][0].view(-1).tolist(), Dataset_nmt.dataset[0][1].view(-1).tolist()
seq1s, seq2s = [Dataset_nmt.input_lang.ix2word[_] for _ in seq1d], [Dataset_nmt.output_lang.ix2word[_] for _ in seq2d]

print(f"{seq1d} --- {seq2d}")
print(f"{seq1s} --- {seq2s}")

#seqs=131072 fra to eng
#voc= 4966 to 3240
[3, 4, 5, 6, 7, 8, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2] --- [3, 4, 5, 6, 7, 8, 9, 1, 2, 2, 2, 2, 2, 2, 2, 2]
['j', 'essaye', 'd', 'arreter', 'tom', '.', 'EOS', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD'] --- ['i', 'm', 'trying', 'to', 'stop', 'tom', '.', 'EOS', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD', 'PAD']


We modify `Encoder` and `Decoder` slightly to take batch_size as a constructor argument. This allows us to reuse these classes with a different batch size later in the assignment.

In [3]:
class Encoder(nn.Module):
    def __init__(self, batch_size:BATCH_SIZE, n_input:int, n_hidden:int):
        super().__init__()
        self.batch_size = batch_size
        self.n_input, self.n_hidden = n_input, n_hidden
        # embedding layer - n_input=vocabulary size, n_hidden=will be embedding size
        self.embedding = nn.Embedding(self.n_input, self.n_hidden, padding_idx=PAD_token)
        # RNN layer, used token by token
        self.rnn_cell = nn.GRU(self.n_hidden, self.n_hidden, batch_first=True)

    # BLUE in the block diagram
    def forward(self, _X, _hn):
        # BS x T x hidden_size
        X_embedded = self.embedding(_X)
        output, hn = self.rnn_cell(X_embedded, _hn)  # RNN iterations inside
        return output, hn

    def init_hidden(self, bs=None):  # 1 x BS x H by design of PyTorch GRU, or T x BS x H
        if bs is None:
            bs = self.batch_size
        return torch.zeros(1, bs, self.n_hidden, device=Device)

class Decoder(nn.Module):
    def __init__(self,  batch_size:BATCH_SIZE, n_hidden:int, n_output:int, dropout:float=0.1):
        super().__init__()
        self.batch_size = batch_size
        self.n_hidden, self.n_output = n_hidden, n_output

        self.embedding = nn.Embedding(self.n_output, self.n_hidden, padding_idx=PAD_token)
        self.dropout = nn.Dropout(dropout)
        self.attn = nn.Linear(in_features=self.n_hidden, out_features=self.n_hidden)  # W_a
        # attention outputs and hn will be cascaded
        self.w_c = nn.Linear(in_features=2*self.n_hidden, out_features=self.n_hidden)  # combine [h_t; c_t]
        self.rnn_cell = nn.GRU(input_size=self.n_hidden, hidden_size=self.n_hidden, batch_first=True)

        # output
        self.w_y = nn.Linear(in_features=self.n_hidden, out_features=self.n_output)

    def forward(self, _X, _hn, _encoder_outputs, enc_len):
        # BS x T x hidden_size
        X_embedded = self.embedding(_X)
        X_embedded = self.dropout(X_embedded)

        # RED in the block diagram
        # hidden state, time t
        rnn_out, hn = self.rnn_cell(X_embedded, _hn)

        # YELLOW in the block diagram, alignment_scores shape:Bx1xT
        alignment_scores = torch.bmm(self.attn(hn.permute(1,0,2)), _encoder_outputs.transpose(1,2))

        # weights, shape:Bx1xT
        attn_weights = nn.functional.softmax(alignment_scores, dim=2)

        # multiplicative attention context vector c_t, shape:Bx1xH
        c_t = torch.bmm(attn_weights, _encoder_outputs)

        # concatenate h_t and the context, shape:Bx1x2H
        hidden_s_t = torch.cat([hn.permute(1,0,2), c_t], dim=2)

        # RED in the block diagram
        # hidden context, shape:Bx1xH
        hidden_s_t = torch.tanh(self.w_c(hidden_s_t))

        # output
        output = nn.functional.log_softmax(self.w_y(hidden_s_t), dim=2)
        # shapes output, hn, attn_w: Bx1xn_output, Bx1xH, Bx1xT
        return output, hn, attn_weights

    def init_hidden(self, bs=None):  # 1 x BS x H by design of PyTorch GRU, or T x BS x H
        if bs is None:
            bs = self.batch_size
        return torch.zeros(1, bs, self.n_hidden, device=Device)

In [4]:
encoder = Encoder(BATCH_SIZE, Dataset_nmt.input_lang.n_words, HIDDEN_N).to(Device)
decoder = Decoder(BATCH_SIZE, HIDDEN_N, Dataset_nmt.output_lang.n_words, dropout=0.1).to(Device)

Dloader_nmt = DataLoader(Dataset_nmt, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

In [5]:
def train(_encoder, _decoder, epochs=3, info=False):
    import sys

    s2s_loss_func = nn.NLLLoss(ignore_index=PAD_token)  # needs (B,C) tensor shape
    e_optim = torch.optim.AdamW(_encoder.parameters(), lr=5e-4)
    d_optim = torch.optim.AdamW(_decoder.parameters(), lr=5e-4)

    _encoder.train()
    _decoder.train()
    N = len(Dloader_nmt)
    totloss = 0
    for e in range(epochs):
        for b, (seq1, seq2) in enumerate(Dloader_nmt):
            seq1 = seq1.to(Device, non_blocking=True)  # x_t
            seq2 = seq2.to(Device, non_blocking=True)  # y_t

            e_hn = _encoder.init_hidden()  # hidden layer - latent space

            e_optim.zero_grad()
            d_optim.zero_grad()

            Nseq1, Nseq2 = seq1.size(1), seq2.size(1)  # input and target lengths

            # Generate BLUE
            e_outputs = torch.zeros(BATCH_SIZE, SEQ_MXLEN, _encoder.n_hidden, device=Device)

            with torch.set_grad_enabled(True):
                # encoder
                for ei in range(Nseq1):
                    e_output, e_hn = _encoder(seq1[:,ei].unsqueeze(1), e_hn)
                    e_outputs[:,ei] = e_output[:, 0]

                # init decoder with GO_token
                d_input = torch.full((BATCH_SIZE, 1), GO_token, device=Device)

                d_hn = e_hn  # hidden layer connection between encoder and decoder, red thick line in the figure

                d_outputs = []

                for di in range(Nseq2):
                    d_output, d_hn, _ = _decoder(d_input, d_hn, e_outputs, Nseq1)
                    d_outputs.append(d_output.squeeze(1))
                    d_input = seq2[:,di].unsqueeze(1)
                    # loss input (B,C) net-output & token ix

                d_outputs = torch.stack(d_outputs, dim=1)  # B x T x n_output
                loss = s2s_loss_func(d_outputs.view(-1, _decoder.n_output), seq2.view(-1))  # B

                loss.backward()
                e_optim.step()
                d_optim.step()

            totloss += loss.item()

            if info:
                sys.stderr.write(f"\rEpoch: {e+1}/{epochs} | Iteration: {b:4d}/{N:4d} | Loss: {totloss:3.2f}")
                sys.stderr.flush()
                totloss = 0

In [6]:
%%time

train(encoder, decoder, epochs=10, info=True)

/Users/jeff/.local/share/mamba/envs/jhu-genai-mod7/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Epoch: 10/10 | Iteration: 2047/2048 | Loss: 0.30

CPU times: user 31min 51s, sys: 21min 48s, total: 53min 39s
Wall time: 27min 12s


We see from the training the loss went down to about 0.30 which indicates a good fit. We'll verify that below.

## Problem 2

In [7]:
def eval_ss(_enc, _dec, _seq1, max_length=SEQ_MXLEN):
    _enc.eval()
    _dec.eval()
    with torch.no_grad():
        Nseq1 = _seq1.size(0)
        _seq1 = _seq1.to(Device)
        e_hn = _enc.init_hidden(bs=1)
        e_outputs = torch.zeros(1, max_length, _enc.n_hidden, device=Device)

        for ei in range(Nseq1):
            e_output, e_hn = _enc(_seq1[ei].view(1,1), e_hn)
            e_outputs[0,ei] = e_output[0, 0]

        d_input = torch.full((1, 1), GO_token, device=Device)

        d_hn = e_hn  # hidden layer connection between encoder and decoder

        d_words = []
        d_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            d_output, d_hn, d_attn = _dec(d_input, d_hn, e_outputs, Nseq1)
            d_attentions[di] = d_attn.data[0]

            # output word index with highest P
            _, topi = d_output.data[0].topk(1)
            if topi.item() != EOS_token:
                d_words.append(Dataset_nmt.output_lang.ix2word[topi.item()])
            else:
                break

            d_input = topi.detach()

        return d_words, d_attentions[:di+1, :Nseq1]  # crop to true encoder length


def eval_rnd(_enc, _dec, n=10):
    import random
    for _ in range(n):
        sample = random.randint(0, len(Dloader_nmt)-1)
        pair = Dataset_nmt.pairs[sample]
        seq1 = Dataset_nmt[sample][0]

        output_words, attentions = eval_ss(_enc, _dec, seq1)

        print(f"seq1:{pair[0]} | seq2:{pair[1]} | result:{' '.join(output_words)}")

## Problem 2

In [10]:
eval_rnd(encoder, decoder, 20)

seq1:tu es precisement la personne a laquelle je veux parler . | seq2:you re just the person i want to speak to . | result:you re just the person i know you want to .
seq1:je viens en juillet . | seq2:i m coming in july . | result:i m coming in july .
seq1:vous etes fort elegante . | seq2:you re very stylish . | result:you re very sophisticated .
seq1:tu effraies les enfants . | seq2:you re scaring the kids . | result:you re scaring the kids .
seq1:c est un homme tres talentueux . | seq2:he s a very talented man . | result:he s a very talented man .
seq1:nous ne sommes pas responsables . | seq2:we re not responsible . | result:we re not responsible .
seq1:je suis desolee mais j ai deja un jules . | seq2:i m sorry but i already have a boyfriend . | result:i m sorry but i already have a boyfriend .
seq1:c est un mordu de cinema . | seq2:he s a movie buff . | result:he s a bit of four age .
seq1:je ne suis pas contente . | seq2:i m not happy . | result:i m not happy .
seq1:je ne suis pas 

In [11]:
eval_rnd(encoder, decoder, 20)

seq1:tu es trop bruyante . | seq2:you re too loud . | result:you re too loud .
seq1:je suis sur que tom sera bientot a la maison . | seq2:i m sure tom will be home soon . | result:i m sure tom will be home soon .
seq1:ils sont armes . | seq2:they re armed . | result:they re armed .
seq1:je ne suis pas convaincu . | seq2:i m not convinced . | result:i m not convinced .
seq1:j essaie de m imaginer ce que vous faites pour vous amuser . | seq2:i m trying to figure out what you do for fun . | result:i m trying to figure out what you do for fun .
seq1:il est assez mignon n est ce pas ? | seq2:he s kind of cute isn t he ? | result:he s old enough aren t he ?
seq1:je suis honnete avec toi . | seq2:i m being honest with you . | result:i m being honest with you .
seq1:je ne m attends pas a un traitement de faveur . | seq2:i m not expecting special treatment . | result:i m not expecting about a hurry !
seq1:elle souffre d une maladie grave . | seq2:she s suffering from a serious disease . | resul

In [12]:
eval_rnd(encoder, decoder, 20)

seq1:nous sommes cinq dans la famille . | seq2:we are a family of five . | result:we are a family of four .
seq1:tu es egoiste . | seq2:you re selfish . | result:you re selfish .
seq1:tu te plains toujours . | seq2:you re always complaining . | result:you re always complaining .
seq1:je suis choque . | seq2:i m shocked . | result:i m shocked .
seq1:il est trop ivre . | seq2:he s too drunk . | result:he s too drunk .
seq1:je vais au commissariat . | seq2:i m going to the police station . | result:i m going to the police station .
seq1:je retourne avec mon ex petite copine . | seq2:i m getting back together with my ex girlfriend . | result:i m getting back together with my ex girlfriend .
seq1:je suis desolee je ne t ai pas entendu . | seq2:i m sorry i didn t hear you . | result:i m sorry i didn t hear you .
seq1:j y retourne . | seq2:i m going back . | result:i m going back .
seq1:il est aussi un vendeur . | seq2:he s a salesman too . | result:he s a bad champion .
seq1:nous ne faisons 

After running the model for 10 epochs and testing random translation sequences several times, the generated outputs are largely the correct English translations. 

This misses are things like:

- family of four instead of family of five
- bad champion instead of salesman too
- malformed but related phrasing like we re just getting on the same

Which are still in the knowledge domain. This indicates the model is not just memorizing one default phrase, but has learned the structure of the task.

## Problem 3

We start by modifying the `eval` function to return predicted token IDs

In [13]:
def eval_ss_ids(_enc, _dec, _seq1, max_length=SEQ_MXLEN):
    _enc.eval()
    _dec.eval()
    with torch.no_grad():
        Nseq1 = _seq1.size(0)
        _seq1 = _seq1.to(Device)
        e_hn = _enc.init_hidden(bs=1)
        e_outputs = torch.zeros(1, max_length, _enc.n_hidden, device=Device)

        for ei in range(Nseq1):
            e_output, e_hn = _enc(_seq1[ei].view(1, 1), e_hn)
            e_outputs[0, ei] = e_output[0, 0]

        d_input = torch.full((1, 1), GO_token, device=Device)
        
        d_hn = e_hn  # hidden layer connection between encoder and decoder

        pred_ids = []

        for di in range(max_length):
            d_output, d_hn, d_attn = _dec(d_input, d_hn, e_outputs, Nseq1)

            _, topi = d_output.data[0].topk(1)
            token_id = topi.item()

            if token_id == EOS_token:
                break

            pred_ids.append(token_id)
            d_input = topi.detach()

        return pred_ids

We create a function to score a single example using PyTorch's `cosine_similarity` function. We have a helper method to convert a sequence to a "bag-of-words" vector.

In [14]:
import torch.nn.functional as F
import numpy as np

def score_one_example(sample_idx, encoder, decoder, dataset):
    seq1, seq2 = dataset[sample_idx]

    pred_ids = eval_ss_ids(encoder, decoder, seq1)

    target_ids = seq2.view(-1).tolist()
    ignore_tokens = {PAD_token, GO_token, EOS_token}
    target_ids = [tid for tid in target_ids if tid not in ignore_tokens]

    vocab_size = dataset.output_lang.n_words

    target_vec = seq_to_bow_vector(target_ids, vocab_size, ignore_tokens=ignore_tokens)
    pred_vec   = seq_to_bow_vector(pred_ids,   vocab_size, ignore_tokens=ignore_tokens)

    score = F.cosine_similarity(target_vec.unsqueeze(0), pred_vec.unsqueeze(0)).item()
    
    target_words = [dataset.output_lang.ix2word[tid] for tid in target_ids]
    pred_words   = [dataset.output_lang.ix2word[tid] for tid in pred_ids]

    return {
        "sample_idx": sample_idx,
        "target": " ".join(target_words),
        "prediction": " ".join(pred_words),
        "cosine_similarity": score
    }

def seq_to_bow_vector(token_ids, vocab_size, ignore_tokens=None):

    vec = np.bincount(token_ids, minlength=vocab_size).astype(np.float32)

    if ignore_tokens:
        for t in ignore_tokens:
            vec[t] = 0

    return torch.tensor(vec, dtype=torch.float32)    

This function iterates over random samples and keeps track of mean and median cosine similarity scores.

In [15]:
import random

def evaluate_cosine_similarity(encoder, decoder, dataset, n_samples=100):
    scores = []

    for _ in range(n_samples):
        sample_idx = random.randint(0, len(dataset) - 1)
        result = score_one_example(sample_idx, encoder, decoder, dataset)
        scores.append(result["cosine_similarity"])

        print(f"target:     {result['target']}")
        print(f"prediction: {result['prediction']}")
        print(f"cosine:     {result['cosine_similarity']:.4f}")
        print("-" * 60)

    mean_score = float(np.mean(scores))
    median_score = float(np.median(scores))

    print(f"\nAverage cosine similarity over {n_samples} samples: {mean_score:.4f}")
    print(f"Median cosine similarity over {n_samples} samples:  {median_score:.4f}")

    return mean_score, median_score, scores

Compute mean and medium for 50 samples

In [16]:
mean_score, median_score, scores = evaluate_cosine_similarity(
    encoder, decoder, Dataset_nmt, n_samples=50
)

target:     she is crying .
prediction: she is wearing a sweater .
cosine:     0.6124
------------------------------------------------------------
target:     i m not a teacher .
prediction: i m not a teacher .
cosine:     1.0000
------------------------------------------------------------
target:     you re the only person that i can trust .
prediction: you re the only person i know that i know .
cosine:     0.7348
------------------------------------------------------------
target:     he s as blind as a bat .
prediction: he s as tall as a bat .
cosine:     0.9000
------------------------------------------------------------
target:     they re babies .
prediction: they re babies .
cosine:     1.0000
------------------------------------------------------------
target:     i m your friend .
prediction: i m your friend .
cosine:     1.0000
------------------------------------------------------------
target:     i m certainly not your friend .
prediction: i m certainly not your friend .


The average cosine similarity of 0.8929 and a median of 1.0 indicate that the predicted responses are highly aligned with the target sentences. A median of 1.0 means that at least half of the evaluated outputs share an identical bag-of-words representation with the expected answer, while the high average suggests that even the remaining predictions retain substantial overlap.

## Problem 4

We define a new class, `QADataset`, which reads in the SQUAD dataset and build seq-to-seq pairs as:

```seq1: <question [EOS]> to seq2: <[GO] answer [EOS]>```

Context is ignored, so we allow for passing in an array of target_titles leaving it to the user of the class to select titles which are logically grouped together.

Question and answer pairs that result in sequences longer than `mxlen` are skipped.

The `normalize_string` method is modified slightly from the original to preserve numbers and some special characters. There are prevalent in the SQUAD dataset and I found through experimentation that keeping them helps with training and the quality of the results. 



In [19]:
import json
import torch
from ep_seq import GO_token, EOS_token, PAD_token, Lang


class QADataset(torch.utils.data.Dataset):

    def __init__(self, json_file, dataset_size:int, mxlen:int, vectorized=False, target_titles=[]):

        super().__init__()

        self.mxlen = mxlen
        self.vectorized = vectorized
        self.target_titles = target_titles

        with open(json_file, encoding="utf-8") as f:
            squad = json.load(f)

        self.input_lang = Lang("qa_in")
        self.output_lang = Lang("qa_out")

        pairs = []

        skipped_answer_too_long = 0
        skipped_seq_too_long = 0
        
        for article in squad["data"]:
            if article["title"] not in target_titles:
                continue
            
            for paragraph in article["paragraphs"]:

                for qa in paragraph["qas"]:

                    if qa["is_impossible"]:
                        continue

                    question = self.normalize_string(qa["question"])
                    answer = self.normalize_string(qa["answers"][0]["text"])

                    # # skip answers with more than 4 tokens to simplify for training
                    # if len(answer.split()) > 3:
                    #     skipped_answer_too_long += 1
                    #     continue

                    seq1 = question
                    seq2 = answer

                    # check we're not greater than the max length
                    if len(seq1.split(" ")) < self.mxlen-1 and len(seq2.split(" ")) < self.mxlen-1:
                        pairs.append([seq1, seq2])
                    else:
                        skipped_seq_too_long += 1         

                    if len(pairs) >= dataset_size:
                        break
                if len(pairs) >= dataset_size:
                    break
            if len(pairs) >= dataset_size:
                break

        print(f"Skipped {skipped_answer_too_long} long answers")
        print(f"Skipped {skipped_seq_too_long} long sequences")
                
        self.pairs = pairs

        # build vocabularies
        for pair in self.pairs:
            self.input_lang.add_sentence(pair[0])
            self.output_lang.add_sentence(pair[1])

        self.dataset = []

        for pair in self.pairs:

            source_ix = [self.input_lang.word2ix[w] for w in pair[0].split(" ")] + [EOS_token]
            target_ix = [self.output_lang.word2ix[w] for w in pair[1].split(" ")] + [EOS_token]

            if self.vectorized:
                source_ix += [PAD_token]*(self.mxlen-len(source_ix))
                target_ix += [PAD_token]*(self.mxlen-len(target_ix))

                source_tensor = torch.tensor(source_ix, dtype=torch.long)
                target_tensor = torch.tensor(target_ix, dtype=torch.long)

            else:
                source_tensor = torch.tensor(source_ix, dtype=torch.long).view(-1,1)
                target_tensor = torch.tensor(target_ix, dtype=torch.long).view(-1,1)

            self.dataset.append((source_tensor, target_tensor))

    @staticmethod
    def normalize_string(s):

        import re
        import unicodedata

        s = ''.join(
            c for c in unicodedata.normalize('NFD', s.lower().strip())
            if unicodedata.category(c) != 'Mn'
        )

        s = re.sub(r"([.!?])", r" \1", s)

        # Change the RegEx from the original to preserve numbers and some special characters since they're prevelant in SQUAD
        s = re.sub(r"[^a-zA-Z0-9.!?' -]+", r" ", s)
        return s

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

The code below creates an instance of the QADataset class using titles that broadly fall under the title "Beyoncé". By selecting Q&A pairs under a common title, we ensure a similar context without having to encode the context into SEQ1, which would slow training. 

- We use a smaller DATASET_SIZE to limit the vocabulary and allow the model to train faster.
- Through experimentation, I found BATCH_SIZE = 16 gave better results than BATCH_SIZE = 32. This is likely because the weights need to be updated more often on the smaller data set.
- SQL_MXLEN=32 is sufficient for the question and answer pairs on Beyoncé.
- HIDDEN_N was changed to 128. I found the larger value allowed the model to better distinguish answers to similar questions.

In [20]:
from torch.utils.data import DataLoader

DATASET_SIZE = 256
BATCH_SIZE = 16
SEQ_MXLEN = 32
HIDDEN_N = 128

Dataset_qa = QADataset(
    "EP_datasets/squad-train-v2.0.json",
    dataset_size=DATASET_SIZE,
    mxlen=SEQ_MXLEN,
    vectorized=True,
    target_titles=[ "Beyoncé"]
)

Dloader_qa = DataLoader(
    Dataset_qa,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

Skipped 0 long answers
Skipped 0 long sequences


Above, we see 256 Q&A pairs would be over the max length of 16 tokens so we've skipped this.

Below, we rename the datasets so we can reuse our original code.

In [21]:
Dataset_nmt = Dataset_qa
Dloader_nmt = Dloader_qa

Sanity check

In [22]:
print("dataset size:", len(Dataset_nmt))
print("batches:", len(Dloader_nmt))
print("input vocab:", Dataset_nmt.input_lang.n_words)
print("output vocab:", Dataset_nmt.output_lang.n_words)

dataset size: 256
batches: 16
input vocab: 581
output vocab: 295


In [23]:
pair = Dataset_nmt.pairs[0]

print("seq1:", pair[0])
print("seq2:", pair[1])

seq1: when did beyonce start becoming popular ?
seq2: in the late 1990s


In [24]:
seq1d, seq2d = Dataset_nmt.dataset[0]

print(seq1d)
print(seq2d)

tensor([3, 4, 5, 6, 7, 8, 9, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2])
tensor([3, 4, 5, 6, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2, 2, 2])


In [25]:
encoder = Encoder(BATCH_SIZE, Dataset_nmt.input_lang.n_words, HIDDEN_N).to(Device)
decoder = Decoder(BATCH_SIZE, HIDDEN_N, Dataset_nmt.output_lang.n_words, dropout=0.1).to(Device)

In [26]:
%%time

train(encoder, decoder, epochs=50, info=True)

Epoch: 50/50 | Iteration:   15/  16 | Loss: 0.53

CPU times: user 1min 6s, sys: 1min 12s, total: 2min 19s
Wall time: 1min 25s


The training loss of 0.53 indicates we likely have convergence.

In the code below, we re-define the `eval_ss` function here so we can play with `SEQ_MXLEN` above. 

We also change `eval_rnd` to use `random.sample` since we're using a fairly narrow dataset.

In [27]:
import random

def eval_ss(_enc, _dec, _seq1, max_length=SEQ_MXLEN):
    _enc.eval()
    _dec.eval()
    with torch.no_grad():
        Nseq1 = _seq1.size(0)
        _seq1 = _seq1.to(Device)
        e_hn = _enc.init_hidden(bs=1)
        e_outputs = torch.zeros(1, max_length, _enc.n_hidden, device=Device)

        for ei in range(Nseq1):
            e_output, e_hn = _enc(_seq1[ei].view(1,1), e_hn)
            e_outputs[0,ei] = e_output[0, 0]

        d_input = torch.full((1, 1), GO_token, device=Device)

        d_hn = e_hn  # hidden layer connection between encoder and decoder

        d_words = []
        d_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            d_output, d_hn, d_attn = _dec(d_input, d_hn, e_outputs, Nseq1)
            d_attentions[di] = d_attn.data[0]

            # output word index with highest P
            _, topi = d_output.data[0].topk(1)
            if topi.item() != EOS_token:
                d_words.append(Dataset_nmt.output_lang.ix2word[topi.item()])
            else:
                break

            d_input = topi.detach()

        return d_words, d_attentions[:di+1, :Nseq1]  # crop to true encoder length

def eval_rnd(_enc, _dec, n=10):
    n = min(n, len(Dataset_nmt))
    samples = random.sample(range(len(Dataset_nmt)), n)
    
    for sample in samples:        
        pair = Dataset_nmt.pairs[sample]
        seq1 = Dataset_nmt[sample][0]

        output_words, attentions = eval_ss(_enc, _dec, seq1)

        print(f"seq1:{pair[0]} | seq2:{pair[1]} | result:{' '.join(output_words)}")

In [28]:
eval_rnd(encoder, decoder, 20)

seq1:how many top five singles came from her first album ? | seq2:four | result:four
seq1:which song did beyonce sing at the first couple's inaugural ball ? | seq2:at last | result:at last
seq1:what was beyonce's role in destiny's child ? | seq2:lead singer | result:lead singer
seq1:in what year did beyonce embark on her dangerously in love tour of europe ? | seq2:november 2003 | result:november 2003
seq1:which of her teachers discovered beyonce's musical talent ? | seq2:dance instructor darlette johnson | result:darlette johnson
seq1:when did beyonce leave destiny's child and become a solo singer ? | seq2:2003 | result:2003
seq1:for which song  did destiny's child take home the grammy award for best r b performance ? | seq2: say my name  | result: say my name 
seq1:how long did the hiatus last ? | seq2:nine months | result:nine months
seq1:independent women part i was on which 2000 film's soundtrack ? | seq2:charlie's angels . | result:charlie's angels .
seq1:which magazine declared h

In [29]:
eval_rnd(encoder, decoder, 20)

seq1:for which song  did destiny's child take home the grammy award for best r b performance ? | seq2: say my name  | result: say my name 
seq1:in which decade did the recording industry association of america recognize beyonce as the the top certified artist ? | seq2:2000s | result:2000s
seq1:who did beyonce marry ? | seq2:jay z . | result:jay z .
seq1:when did beyonce release dangerously in love ? | seq2:2003 | result:2003
seq1:who beat out beyonce for best female video  ? | seq2:taylor swift | result:taylor swift
seq1:what was beyonce's role in destiny's child ? | seq2:lead singer | result:lead singer
seq1:what was the first album beyonce released as a solo artist ? | seq2:dangerously in love | result:dangerously in love
seq1:what was the name of beyonce's first international tour ? | seq2:the beyonce experience | result:the beyonce experience
seq1:how did the critics view the movie  ''the fighting temptations'' ? | seq2:mixed reviews | result:mixed reviews
seq1:who did they tie wit

In [30]:
eval_rnd(encoder, decoder, 20)

seq1:her first appearance performing since giving birth was where ? | seq2:revel atlantic city's ovation hall | result:new york's roseland ballroom
seq1:how many records did beyonce sell as part of destiny's child ? | seq2:60 million | result:60 million
seq1:which organization did beyonce donate her pay for the private performance to ? | seq2:clinton bush haiti fund . | result:clinton bush haiti fund .
seq1:when did beyonce begin her second world tour ? | seq2:march 2009 | result:march 2009
seq1:what major event did beyonce perform at on february 1  2004 ? | seq2:super bowl xxxviii | result:super bowl xxxviii
seq1:destiny's child song  killing time  was included in which film's soundtrack ? | seq2:men in black . | result:men in black .
seq1:how many awards was beyonce nominated for at the 52nd grammy awards ? | seq2:ten | result:ten
seq1:beyonce's group changed their name to destiny's child in what year ? | seq2:1996 | result:1996
seq1:what musical comedy did beyonce star in along with

We see the model gets most of the questions correct in this random sample. Where it misses, it is still answering with valid Beyonce knowledge, e.g.:

- atlantic city's ovation hall instead of new york's roseland ballroom

The model learned a substantial number of question-answer mappings, though it still struggles to distinguish a few similar facts.

Now we check the cosine similarity. We redeclare eval_ss_ids because we changed `SQ_MXLEN` above.

In [32]:
def eval_ss_ids(_enc, _dec, _seq1, max_length=SEQ_MXLEN):
    _enc.eval()
    _dec.eval()
    with torch.no_grad():
        Nseq1 = _seq1.size(0)
        _seq1 = _seq1.to(Device)
        e_hn = _enc.init_hidden(bs=1)
        e_outputs = torch.zeros(1, max_length, _enc.n_hidden, device=Device)

        for ei in range(Nseq1):
            e_output, e_hn = _enc(_seq1[ei].view(1, 1), e_hn)
            e_outputs[0, ei] = e_output[0, 0]

        d_input = torch.full((1, 1), GO_token, device=Device)
        
        d_hn = e_hn  # hidden layer connection between encoder and decoder

        pred_ids = []

        for di in range(max_length):
            d_output, d_hn, d_attn = _dec(d_input, d_hn, e_outputs, Nseq1)

            _, topi = d_output.data[0].topk(1)
            token_id = topi.item()

            if token_id == EOS_token:
                break

            pred_ids.append(token_id)
            d_input = topi.detach()

        return pred_ids

In [34]:
mean_score, median_score, scores = evaluate_cosine_similarity(
    encoder, decoder, Dataset_nmt, n_samples=50
)

target:     dwayne wiggins's grass roots entertainment
prediction: dwayne wiggins's grass roots entertainment
cosine:     1.0000
------------------------------------------------------------
target:     cadillac records
prediction: cadillac records
cosine:     1.0000
------------------------------------------------------------
target:     st . john's united methodist church
prediction: st . john's united methodist church
cosine:     1.0000
------------------------------------------------------------
target:     mariah carey
prediction: mariah carey
cosine:     1.0000
------------------------------------------------------------
target:     missy elliott and alicia keys
prediction: missy elliott and alicia keys
cosine:     1.0000
------------------------------------------------------------
target:     sony music
prediction: sony music
cosine:     1.0000
------------------------------------------------------------
target:     dwayne wiggins's grass roots entertainment
prediction: dwayne wi

The average cosine similarity of 0.9183 and a median of 1.0 indicate that the predicted responses are highly aligned with the target sentences. A median of 1.0 means that at least half of the evaluated outputs share an identical bag-of-words representation with the expected answer, while the high average suggests that even the remaining predictions retain substantial overlap. This is a good fit.

## Problem 5

# Research Project Status Report

**Project Title:** Optimization Log Copilot: Using Large Language Models to Analyze Optimization Solver Logs  

---

# Project Overview

The goal of my research project is to explore how large language models (LLMs) can assist in analyzing solver logs generated by mixed-integer programming (MIP) solvers. Modern optimization solvers produce detailed logs describing the solving process, including information about branching decisions, cutting planes, node exploration, heuristics, and optimality gap progression.

These logs can be extremely long and complex, and interpreting them often requires significant expertise in optimization algorithms. The central idea of this project is to build an Optimization Log Copilot, a system that uses LLMs to interpret solver logs and provide diagnostic insights about solver performance.

The system aims to answer questions such as:

- Why is the solver exploring a large number of nodes?
- Is the root LP relaxation weak?
- Are cutting planes improving the bound?
- What structural issues might exist in the optimization model?

By automating portions of solver log analysis, this tool could help both researchers and practitioners better understand solver behavior and improve optimization model performance.

---

# Project Development

## Dataset

I have collected about 200 solver logs across a range of problems from the MIPLIB 2017 (https://miplib.zib.de/) set as well as from a series of models I created that represent Ramsey problems (https://en.wikipedia.org/wiki/Ramsey%27s_theorem), e.g., R(4,4;15). These include logs that have optimal solutions, infeasible solutions, various cutting planes and heuristics, difficult root relaxations, varying optimality gap closures, a wide variety of node counts, and various degenerate problems.   

For prototype development, I'm working with a subset of smaller logs which I'm using to test parsing and summarization approaches.

I have an automated way to run the solvers Rose, Gurobi, HiGHS, and CPLEX which allows me to generate additional logs as needed.

---

## Log Parsing and Representation

Solver logs are semi-structured text so a preprocessing step is required to convert them into a structured format that can be analyzed by the model.

I've built a prototype pipeline that includes:

1. Raw solver log input  
2. Parsing and extraction of structured events  
3. Representation of events in JSON format  
4. Feeding structured information into the model 

I am able to associate various metrics available from the solvers, (e.g., node counts, optimality status, bound updates) with each of the logs. This representation should allow the solution to analyze solver progress more effectively than using raw text logs alone.

I am working on adding natural language explanations for the logs to the training data. My intent is to have the training data consist of "triples" of the form:

(log_text, diagnostic_labels, explanation)

This will allow training both a transformer-based encoder that process raw solver logs and predicts bottleneck labels as well as a text generation model that produces a natural-language explanation of the log.
---

## Method

The system will combine transformer-based sequence understanding with generative language models to produce both structured predictions and natural-language explanations.

![Research Project Diagram](research_project_diagram.png)

### Log Preprocessing and Dataset Construction

Each solver run is converted into a training example consisting of a triple:

(log_text, diagnostic_labels, explanation)

where log_text is the cleaned solver log, diagnostic_labels represent bottleneck categories (e.g., weak relaxation, excessive branching, ineffective cuts, degeneracy), and explanation is a natural-language summary describing solver behavior and potential improvements.

Right now, I have some heuristic rules to generate the initial labels and am then looking at them and making tweaks manually. Most of the explanations are being done manually.

### Transformer-Based Log Understanding

My intent is to use a transformer encoder to generate contextual representations of the log text. Since these are long, I'm looking at using BERT-style encoders, Longformer, or BigBird.


### Bottleneck Classification

The learning task here is multi-label classification of solver performance issues. Possible categories include weak LP relaxation, excessive branching, ineffective cut generation, and numerical instability.

The classifier will predict bottleneck probabilities, and I'm looking at using a sigmoid output layer for this.

Performance will be evaluated using metrics such as precision, recall, and F1 score.

### Generative Explanation Model

To produce human-readable diagnostics, I plan to use a sequence-to-sequence generative model such as T5 or FLAN-T5 to generate explanations conditioned on the solver log and predicted diagnostic labels.

The model will be trained to generate concise summaries describing solver behavior and recommend potential improvements to model formulation or solver configuration.

---

# Next Steps


## Experimental Design

The experimental section will evaluate the following capabilities:

- accuracy of solver log summaries  
- ability of the LLM to identify solver performance issues  
- usefulness of generated diagnostic recommendations  

Evaluation will involve comparing LLM-generated explanations with expected solver behavior patterns.

---

## Dataset Expansion

More solver logs will be collected from benchmark optimization runs. These logs will allow the system to be tested across a range of solver behaviors, including cases with large optimality gaps, heavy branching, and different solver strategies.

---

## System Improvements

Planned improvements include:

- improving log parsing and event extraction  
- experimenting with retrieval-augmented generation  
- refining prompts to improve diagnostic reasoning  

---

# Summary

The literature review is largely complete, and I'm working on a basic notebook prototype developed to process sample solver logs.

The remaining work will focus on expanding the dataset, improving the model interaction pipeline, and conducting experiments to evaluate the system's effectiveness.

---

# References

Akhtar, S., Khan, S., & Parkinson, S. (2025). "LLM-based Event Log Analysis Techniques: A Survey." arXiv preprint arXiv:2502.00677.

C. Raffel et al., “Exploring the limits of transfer learning with a unified text-to-text transformer,” JMLR, 2020.

H. W. Chung et al., “Scaling instruction-finetuned language models,” arXiv:2210.11416, 2022.

J. Devlin et al., “BERT: Pre-training of deep bidirectional transformers for language understanding,” NAACL, 2019.

F. Pedregosa et al., “Scikit-learn: Machine learning in Python,” JMLR, 2011.